In [1]:
import sys
import os

# Add the root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(project_root)

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy import stats
from src.utils import calculate_rsi, calculate_macd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [3]:
# Load dataset
df = pd.read_csv(r'..\..\data\bitcoin\btcusd_1-min_data.csv')
# Convert 'Timestamp' to datetime and set as index
df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='s')
df.set_index('Timestamp', inplace=True)
df = df.sort_index()
df.tail() 

,Open,High,Low,Close,Volume
Timestamp,,,,,
2025-08-02 21:50:00,112955.0,112955.0,112943.0,112944.0,0.369770
2025-08-02 21:51:00,112944.0,112944.0,112943.0,112944.0,0.009608
2025-08-02 21:52:00,112944.0,112956.0,112905.0,112917.0,1.211315
2025-08-02 21:53:00,112917.0,112971.0,112917.0,112971.0,0.366485
2025-08-02 21:54:00,112971.0,113025.0,112970.0,113025.0,2.309270


In [5]:
# Compute the Average price
df['Average_Price'] = (df['Open'] + df['High'] + df['Low'] + df['Close']) / 4

In [6]:
# Compute VWAP and Resample
VWP = df['Volume'] * df['Average_Price']
# Resample VWAP to daily, monthly, yearly, and quarterly frequency
vwap_minutely = VWP.resample('min').sum() / df['Volume'].resample('min').sum()
vwap_daily = VWP.resample('D').sum()/ df['Volume'].resample('D').sum()
vwap_monthly = VWP.resample('ME').sum()/ df['Volume'].resample('ME').sum()
vwap_yearly = VWP.resample('YE-DEC').sum()/ df['Volume'].resample('YE-DEC').sum()
vwap_quarterly = VWP.resample('QE-DEC').sum()/ df['Volume'].resample('QE-DEC').sum()

df_minute = vwap_minutely.to_frame(name='Weighted_Price')
df_daily = vwap_daily.to_frame(name='Weighted_Price')
df_monthly = vwap_monthly.to_frame(name='Weighted_Price')
df_yearly = vwap_yearly.to_frame(name='Weighted_Price')
df_quarterly = vwap_quarterly.to_frame(name='Weighted_Price')